<a href="https://colab.research.google.com/github/NileshPatil24-a/Deep_Learning/blob/main/cat_v_Dong.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [1]:
!mkdir -p ~/.kaggle
!cp kaggle.json ~/.kaggle/
!chmod 600 ~/.kaggle/kaggle.json


In [2]:
!kaggle datasets download -d bhavikjikadara/dog-and-cat-classification-dataset


Dataset URL: https://www.kaggle.com/datasets/bhavikjikadara/dog-and-cat-classification-dataset
License(s): apache-2.0
100% 772M/775M [00:03<00:00, 154MB/s] 
100% 775M/775M [00:03<00:00, 212MB/s]


In [3]:
import zipfile

zip_ref = zipfile.ZipFile('/content/dog-and-cat-classification-dataset.zip', 'r')
zip_ref.extractall('/content')
zip_ref.close()


In [4]:
import os
from PIL import Image

def remove_corrupted_images(folder):
    for category in ["Cat", "Dog"]:
        path = os.path.join(folder, category)
        for file in os.listdir(path):
            file_path = os.path.join(path, file)
            try:
                img = Image.open(file_path)
                img.verify()
            except:
                os.remove(file_path)

remove_corrupted_images("/content/PetImages")


/usr/local/lib/python3.12/dist-packages/PIL/TiffImagePlugin.py:950: UserWarning: Truncated File Read
  warnings.warn(str(msg))


In [5]:
import os
import shutil
import random

source_dir = "/content/PetImages"
base_dir = "/content/data"

train_dir = os.path.join(base_dir, "train")
test_dir  = os.path.join(base_dir, "test")

os.makedirs(train_dir, exist_ok=True)
os.makedirs(test_dir, exist_ok=True)

for category in ["Cat", "Dog"]:
    os.makedirs(os.path.join(train_dir, category), exist_ok=True)
    os.makedirs(os.path.join(test_dir, category), exist_ok=True)

    files = os.listdir(os.path.join(source_dir, category))
    files = [f for f in files if f.endswith(".jpg")]
    random.shuffle(files)

    split = int(0.8 * len(files))   # 80% train, 20% test

    for f in files[:split]:
        shutil.copy(
            os.path.join(source_dir, category, f),
            os.path.join(train_dir, category, f)
        )

    for f in files[split:]:
        shutil.copy(
            os.path.join(source_dir, category, f),
            os.path.join(test_dir, category, f)
        )


In [6]:
import tensorflow
from tensorflow  import keras
from keras import Sequential
from keras.layers import Dense, Flatten, Conv2D,MaxPooling2D

In [7]:
# divide into the batch_size

train_ds = keras.utils.image_dataset_from_directory(
    directory='/content/data/train',
    labels='inferred',
    label_mode ='int',
    batch_size=32,
    image_size=(256,256)
)

validation_ds = keras.utils.image_dataset_from_directory(
    directory='/content/data/test',
    labels='inferred',
    label_mode ='int',
    batch_size=32,
    image_size=(256,256)
)

Found 19998 files belonging to 2 classes.
Found 5000 files belonging to 2 classes.


In [8]:

# Normalize
def process(image, label):
    image = tensorflow.cast(image/255., tensorflow.float32)
    return image, label

train_ds = train_ds.map(process)
validation_ds = validation_ds.map(process)


In [9]:
# CNN Model

model = Sequential()
model.add(Conv2D(32, kernel_size=(3,3), activation='relu', padding='valid', input_shape=(256,256,3), ))
model.add(MaxPooling2D(pool_size=(2,2), strides=2, padding='valid'))

model.add(Conv2D(64, activation='relu', padding='valid', kernel_size=(3,3)))
model.add(MaxPooling2D(pool_size=(2,2), strides=2, padding='valid'))

model.add(Conv2D(128, activation='relu', padding='valid', kernel_size=(3,3)))
model.add(MaxPooling2D(pool_size=(2,2), strides=2, padding='valid'))


model.add(Flatten())

model.add(Dense(128, activation='relu'))
model.add(Dense(64, activation='relu'))
model.add(Dense(1, activation='sigmoid'))


/usr/local/lib/python3.12/dist-packages/keras/src/layers/convolutional/base_conv.py:113: UserWarning: Do not pass an `input_shape`/`input_dim` argument to a layer. When using Sequential models, prefer using an `Input(shape)` object as the first layer in the model instead.
  super().__init__(activity_regularizer=activity_regularizer, **kwargs)


In [10]:
model.summary()

Model: "sequential"

┏━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━┓
┃ Layer (type)                    ┃ Output Shape           ┃       Param # ┃
┡━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━┩
│ conv2d (Conv2D)                 │ (None, 254, 254, 32)   │           896 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ max_pooling2d (MaxPooling2D)    │ (None, 127, 127, 32)   │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ conv2d_1 (Conv2D)               │ (None, 125, 125, 64)   │        18,496 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ max_pooling2d_1 (MaxPooling2D)  │ (None, 62, 62, 64)     │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ conv2d_2 (Conv2D)               │ (None, 60, 60, 128)    │        73,856 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ max_pooling2d_2 (MaxPooling2D)  │ (None, 30, 30, 128)    │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ flatten (Flatten)               │ (None, 115200)         │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dense (Dense)                   │ (None, 128)            │    14,745,728 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dense_1 (Dense)                 │ (None, 64)             │         8,256 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dense_2 (Dense)                 │ (None, 1)              │            65 │
└─────────────────────────────────┴────────────────────────┴───────────────┘

 Total params: 14,847,297 (56.64 MB)

 Trainable params: 14,847,297 (56.64 MB)

 Non-trainable params: 0 (0.00 B)

In [11]:
model.compile(optimizer='adam', loss='binary_crossentropy',  metrics=['accuracy'])

In [12]:
model.fit(train_ds, epochs=10, validation_data=validation_ds)

Epoch 1/10
147/625 ━━━━━━━━━━━━━━━━━━━━ 34s 73ms/step - accuracy: 0.5286 - loss: 0.7406

InvalidArgumentError: Graph execution error:

Detected at node decode_image/DecodeImage defined at (most recent call last):
<stack traces unavailable>
Detected at node decode_image/DecodeImage defined at (most recent call last):
<stack traces unavailable>
2 root error(s) found.
  (0) INVALID_ARGUMENT:  Unknown image file format. One of JPEG, PNG, GIF, BMP required.
	 [[{{node decode_image/DecodeImage}}]]
	 [[IteratorGetNext]]
	 [[IteratorGetNext/_2]]
  (1) INVALID_ARGUMENT:  Unknown image file format. One of JPEG, PNG, GIF, BMP required.
	 [[{{node decode_image/DecodeImage}}]]
	 [[IteratorGetNext]]
0 successful operations.
0 derived errors ignored. [Op:__inference_multi_step_on_iterator_2436]

In [13]:
import os
from PIL import Image

def clean_folder(folder):
    for root, dirs, files in os.walk(folder):
        for file in files:
            path = os.path.join(root, file)
            try:
                img = Image.open(path)
                img.verify()
            except:
                os.remove(path)

clean_folder("/content/data/train")
clean_folder("/content/data/test")


/usr/local/lib/python3.12/dist-packages/PIL/TiffImagePlugin.py:950: UserWarning: Truncated File Read
  warnings.warn(str(msg))


In [14]:
from tensorflow import keras

train_ds = keras.utils.image_dataset_from_directory(
    "/content/data/train",
    image_size=(256,256),
    batch_size=32
)

validation_ds = keras.utils.image_dataset_from_directory(
    "/content/data/test",
    image_size=(256,256),
    batch_size=32
)


Found 19998 files belonging to 2 classes.
Found 5000 files belonging to 2 classes.


In [15]:
import tensorflow as tf

train_ds = train_ds.map(lambda x, y: (x/255.0, y))
validation_ds = validation_ds.map(lambda x, y: (x/255.0, y))


In [16]:
model.fit(train_ds, epochs=10, validation_data=validation_ds)


Epoch 1/10
 15/625 ━━━━━━━━━━━━━━━━━━━━ 44s 73ms/step - accuracy: 0.5797 - loss: 0.6733

InvalidArgumentError: Graph execution error:

Detected at node decode_image/DecodeImage defined at (most recent call last):
<stack traces unavailable>
Detected at node decode_image/DecodeImage defined at (most recent call last):
<stack traces unavailable>
2 root error(s) found.
  (0) INVALID_ARGUMENT:  Number of channels inherent in the image must be 1, 3 or 4, was 2
	 [[{{node decode_image/DecodeImage}}]]
	 [[IteratorGetNext]]
	 [[IteratorGetNext/_2]]
  (1) INVALID_ARGUMENT:  Number of channels inherent in the image must be 1, 3 or 4, was 2
	 [[{{node decode_image/DecodeImage}}]]
	 [[IteratorGetNext]]
0 successful operations.
0 derived errors ignored. [Op:__inference_multi_step_on_iterator_2436]